# CivicLens: Model Evaluation & Visual Validation

This notebook loads the trained YOLO model weights, runs evaluation against the validation split, and visualizes predictions on randomly chosen test images.

In [ ]:
# 1. Setup workspace
%cd /kaggle/working/civiclens/ml-engine

In [ ]:
# 2. Run standard evaluation
from src.evaluation.evaluate import evaluate_model
from src.utils.config import load_config
from pathlib import Path

config = load_config()
best_weights = Path(config["workspace"]["output_dir"]) / "runs" / "yolo11m_run" / "weights" / "best.pt"
yolo_yaml = Path(config["workspace"]["unified_data_dir"]) / "dataset.yaml"

if best_weights.exists() and yolo_yaml.exists():
    results = evaluate_model(best_weights, yolo_yaml, config)
else:
    print("Trained weights or dataset config not found. Make sure training completed first.")

In [ ]:
# 3. Plot sample predictions visually
import random
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

if best_weights.exists():
    model = YOLO(best_weights)
    # Get validation images
    val_images = glob.glob(f"{config['workspace']['unified_data_dir']}/images/val/*")
    if val_images:
        img_path = random.choice(val_images)
        pred_res = model(img_path)
        annotated_img = pred_res[0].plot() # YOLO returns BGR image
        
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("Visual Validation Prediction Output")
        plt.show()
    else:
        print("No validation images found in unified dataset path.")
else:
    print("No weights found.")